# 🚗 ATVED — Train License Plate DETECTION Model (V2)

**IMPORTANT: This trains a DETECTION model (single class, bounding boxes around plates).**  
This is NOT a classification/recognition model. The model learns to FIND plates in an image, not READ them.

### Setup Checklist
1. **GPU Runtime:** `Runtime → Change runtime type → T4 GPU`
2. **Dataset:** You will paste a Roboflow download snippet (see Step 2)
3. **Training time:** ~20-30 minutes on T4 GPU (50 epochs)
4. **Output:** `plate_detector_v2_best.pt` downloaded to your computer

In [ ]:
# ============================================================
# Step 0: Install dependencies & verify GPU
# ============================================================
!pip install ultralytics roboflow -q

import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
else:
    print("⚠️ NO GPU! Go to Runtime → Change runtime type → T4 GPU")
    raise RuntimeError("GPU required for training")

## Step 1: Download a License Plate DETECTION Dataset

### How to get the dataset:
1. Go to one of these Roboflow datasets (pick ONE):
   - **Option A (Recommended, ~4k images):** [Indian License Plate Detection](https://universe.roboflow.com/license-plate-detection-khhkb/indian-license-plate-detection-6tmbr)
   - **Option B (~20k images, larger):** [Indian Number Plates Object Detection](https://universe.roboflow.com/yolov8/indian-number-plates-object-detection-dataset)
2. Click **"Download Dataset"** (or **"Export"**)
3. Select format: **YOLOv8**
4. Select **"Show download code"**
5. **Copy the Python code snippet** and paste it into the cell below

### ⚠️ CRITICAL: Verify the dataset is for DETECTION
On the dataset page, check:
- Task type says **"Object Detection"** (NOT Classification, NOT Instance Segmentation)
- Classes section shows 1 class like `license_plate` or `number_plate` (NOT hundreds of plate numbers)
- Images show bounding boxes drawn AROUND plates

In [ ]:
# ============================================================
# Step 2: PASTE YOUR ROBOFLOW DOWNLOAD CODE HERE
# ============================================================
# DELETE everything below and paste the code from Roboflow.
# It will look something like:
#
# from roboflow import Roboflow
# rf = Roboflow(api_key="YOUR_API_KEY")
# project = rf.workspace("...").project("...")
# version = project.version(1)
# dataset = version.download("yolov8")

raise RuntimeError("\n\n❌ STOP! Paste your Roboflow download code above this line!\n")

In [ ]:
# ============================================================
# Step 3: Verify the dataset is correct BEFORE training
# ============================================================
import os
import yaml
import glob

data_yaml_path = os.path.join(dataset.location, "data.yaml")
with open(data_yaml_path, 'r') as f:
    data_config = yaml.safe_load(f)

class_names = data_config.get('names', {})
num_classes = data_config.get('nc', len(class_names))

print("=" * 60)
print("DATASET VERIFICATION REPORT")
print("=" * 60)
print(f"Location:      {dataset.location}")
print(f"Num classes:   {num_classes}")
print(f"Class names:   {class_names}")

# Count images
train_imgs = glob.glob(os.path.join(dataset.location, 'train', 'images', '*'))
val_imgs = glob.glob(os.path.join(dataset.location, 'valid', 'images', '*'))
test_imgs = glob.glob(os.path.join(dataset.location, 'test', 'images', '*'))
print(f"Train images:  {len(train_imgs)}")
print(f"Val images:    {len(val_imgs)}")
print(f"Test images:   {len(test_imgs)}")
print(f"Total images:  {len(train_imgs) + len(val_imgs) + len(test_imgs)}")

# Verify labels have bounding box format (class x_center y_center width height)
train_labels = glob.glob(os.path.join(dataset.location, 'train', 'labels', '*.txt'))
if train_labels:
    with open(train_labels[0], 'r') as f:
        sample_label = f.read().strip()
    print(f"\nSample label:  {train_labels[0]}")
    print(f"Content:       {sample_label}")
    
    # Check format: each line should have 5 numbers (class x y w h)
    parts = sample_label.split()
    if len(parts) == 5:
        print("\n\u2705 Label format looks correct (class x_center y_center width height)")
    else:
        print(f"\n\u26a0\ufe0f Label has {len(parts)} values, expected 5")

# CRITICAL CHECK: Reject classification datasets
if num_classes > 10:
    print("\n" + "\u274c" * 20)
    print(f"DANGER: {num_classes} classes detected!")
    print("This looks like a CLASSIFICATION dataset (one class per plate number).")
    print("You need a DETECTION dataset with 1-2 classes (e.g. 'license_plate').")
    print("DO NOT PROCEED. Go back and pick a different dataset.")
    print("\u274c" * 20)
    raise ValueError(f"Dataset has {num_classes} classes - this is a classification dataset, not detection!")
else:
    print("\n\u2705 Class count looks correct for a detection dataset.")
    print("\u2705 Dataset is ready for training!")

In [ ]:
# ============================================================
# Step 4: Train YOLOv8n (Nano) — Plate Detection
# ============================================================
# Budget: 50 epochs, ~20-30 min on T4
# Model: yolov8n.pt (nano — fastest, fits in any GPU)
# ============================================================

from ultralytics import YOLO
import time

model = YOLO("yolov8n.pt")

start_time = time.time()

results = model.train(
    data=data_yaml_path,
    epochs=50,
    imgsz=640,
    batch=16,
    name="atved_plate_detector_v2",
    save=True,
    verbose=True,
    patience=10,       # early stopping if no improvement for 10 epochs
    save_period=10,    # checkpoint every 10 epochs
)

elapsed = time.time() - start_time
print(f"\n\u2705 Training complete in {elapsed/60:.1f} minutes")

In [ ]:
# ============================================================
# Step 5: Report Metrics
# ============================================================
import csv

results_csv = "runs/detect/atved_plate_detector_v2/results.csv"

if os.path.exists(results_csv):
    with open(results_csv, 'r') as f:
        reader = csv.DictReader(f)
        rows = list(reader)
    
    # Get the last (best) epoch's metrics
    last = rows[-1]
    
    # Clean up column names (they have leading spaces)
    last = {k.strip(): v.strip() for k, v in last.items()}
    
    print("=" * 60)
    print("FINAL TRAINING METRICS")
    print("=" * 60)
    print(f"Precision:     {float(last.get('metrics/precision(B)', 0)):.4f}")
    print(f"Recall:        {float(last.get('metrics/recall(B)', 0)):.4f}")
    print(f"mAP@50:        {float(last.get('metrics/mAP50(B)', 0)):.4f}")
    print(f"mAP@50-95:     {float(last.get('metrics/mAP50-95(B)', 0)):.4f}")
    print("=" * 60)
    
    # Sanity check
    map50 = float(last.get('metrics/mAP50(B)', 0))
    if map50 > 0.7:
        print(f"\n\u2705 mAP@50 = {map50:.4f} \u2014 Model looks good!")
    elif map50 > 0.5:
        print(f"\n\u26a0\ufe0f mAP@50 = {map50:.4f} \u2014 Acceptable but not great. Consider more epochs.")
    else:
        print(f"\n\u274c mAP@50 = {map50:.4f} \u2014 Poor performance. Check dataset quality.")
else:
    print("results.csv not found, checking for metrics in other locations...")
    !find runs/ -name 'results.csv' 2>/dev/null

In [ ]:
# ============================================================
# Step 6: Quick visual test on a sample image
# ============================================================
from IPython.display import Image, display
import glob

# Show some validation predictions
val_preds = glob.glob("runs/detect/atved_plate_detector_v2/val_batch*_pred.jpg")
if val_preds:
    print("Validation batch predictions:")
    for vp in val_preds[:2]:
        display(Image(filename=vp, width=800))
else:
    print("No validation prediction images found")

# Also run inference on a val image to show detection confidence
best_model = YOLO("runs/detect/atved_plate_detector_v2/weights/best.pt")
val_images = glob.glob(os.path.join(dataset.location, 'valid', 'images', '*'))
if val_images:
    test_result = best_model.predict(val_images[0], conf=0.25, verbose=False)
    for box in test_result[0].boxes:
        conf = float(box.conf[0])
        cls = int(box.cls[0])
        print(f"  Detected: class={best_model.names[cls]}, confidence={conf:.4f}")

In [ ]:
# ============================================================
# Step 7: Download the trained model
# ============================================================
from google.colab import files
import shutil

src = "runs/detect/atved_plate_detector_v2/weights/best.pt"
dst = "plate_detector_v2_best.pt"
shutil.copy(src, dst)

print(f"Model size: {os.path.getsize(dst) / 1e6:.1f} MB")
print(f"\n\u2705 Model saved as: {dst}")
print()
print("INSTRUCTIONS:")
print("1. The file will download to your browser's Downloads folder")
print("2. Move it to: ATVED_Gridlock/computer_vision_models/plate_detector_v2_best.pt")
print("3. The demo script will automatically pick it up")
print()

files.download(dst)